# Cross-Species Evolutionary Heatmaps — All Cell Types

Runs the evolutionary accessibility module analysis for every cell type
that has robust DESeq2 results, producing:
- Per-cell-type blocked heatmaps (PDF + PNG)
- Per-cell-type BED files for each evolutionary module (full candidate lists)
- SNC (Substituted Nucleotide Change) enrichment analysis for human-upregulated modules

All reusable functions live in `src/evolutionary_heatmap.py`.

In [1]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.stats.multitest import multipletests
import pybedtools

from src.evolutionary_heatmap import (
    is_true,
    sanitize_name,
    load_pseudobulk_matrices,
    build_regions,
    build_heatmap_matrix,
    plot_evolutionary_heatmap,
    export_beds,
    load_snc_bedtool,
    build_snc_backgrounds,
    compute_snc_enrichment,
    plot_snc_enrichment_dotplot,
    discover_cell_types,
)

print("Imports OK")

Imports OK


In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE       = "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
CONS_DIR   = f"{BASE}/cross_species_consensus_v3"
DESEQ_DIR  = f"{CONS_DIR}/13_deseq2_R_pseudobulk"
ANNO_PATH  = f"{CONS_DIR}/07_master_annotation/master_annotation_corrected.tsv"
FRAG_DIR   = f"{CONS_DIR}/12_fragment_matrices"
ROBUST_DIR = f"{DESEQ_DIR}/differential_results/ultra_robust_branches"
BRANCH_DIR = f"{DESEQ_DIR}/differential_results/evolutionary_branches"
SNC_PATH   = ("/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine"
               "/dl_models/data/SNC/Prufer_SNC_hg38_full.bed")

SPECIES = ["Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset"]
MAX_PER_BLOCK = 50

# ── Heatmap view ──────────────────────────────────────────────────────────────
# Options: "species" | "donor" | "donor_region"
ACCESSIBILITY_VIEW = "donor_region"
ROW_ZSCORE = False

# ── Output folders ─────────────────────────────────────────────────────────────
OUT_HEATMAPS = Path(DESEQ_DIR) / "heatmaps"
OUT_BEDS     = Path(DESEQ_DIR) / "region_beds"
OUT_ENRICH   = Path(DESEQ_DIR) / "enrichment" / "SNC"

for d in [OUT_HEATMAPS, OUT_BEDS, OUT_ENRICH]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output roots:")
print(f"  Heatmaps  : {OUT_HEATMAPS}")
print(f"  BED files : {OUT_BEDS}")
print(f"  Enrichment: {OUT_ENRICH}")

Output roots:
  Heatmaps  : /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/heatmaps
  BED files : /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/region_beds
  Enrichment: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/enrichment/SNC


In [3]:
# ── Load shared resources ─────────────────────────────────────────────────────

print("Loading master annotation...")
anno = pd.read_csv(ANNO_PATH, sep="\t", low_memory=False).set_index("peak_id")
orth_cols = {sp: f"{sp}_orth" for sp in SPECIES}
det_cols  = {sp: f"{sp}_det"  for sp in SPECIES}
print(f"  {anno.shape[0]:,} peaks × {anno.shape[1]} columns")

# Discover cell types with robust results
CELL_TYPES = discover_cell_types(ROBUST_DIR, FRAG_DIR, SPECIES)
print(f"\nCell types with robust results ({len(CELL_TYPES)}):")
for ct in CELL_TYPES:
    print(f"  {ct}")

# ── Contrast-specific SNC backgrounds ─────────────────────────────────────────
# Each module is tested against the peaks detectable in its relevant species,
# matching the DESeq2 contrast design.
snc_backgrounds = build_snc_backgrounds(anno, det_cols, SPECIES)
print(f"\nSNC backgrounds (peaks detected in relevant species):")
for module, bg in snc_backgrounds.items():
    print(f"  {module:<35} {len(bg):>8,} peaks")

# ── Load SNC BedTool (fixed modern-human-specific SNCs only) ──────────────────
print("\nLoading SNC BED file...")
snc_bt = load_snc_bedtool(SNC_PATH, filter_fixed=True)
print(f"  {len(snc_bt):,} fixed SNCs loaded")

Loading master annotation...
  1,142,003 peaks × 59 columns

Cell types with robust results (13):
  Colonocytes
  Crypt Fibroblasts WNT2B+
  ECs
  Enterocytes
  Goblet cells
  Macrophages
  Naive B cells
  Plasma B cells
  Specialized Fibroblasts RSPO3+ only
  Specialized Fibroblasts SYNM+
  Stem cells
  T cells
  TA cells

SNC backgrounds (peaks detected in relevant species):
  Human-specific DNA                   629,491 peaks
  Human UP vs All                       64,058 peaks
  ILS: Human + Gorilla UP              121,016 peaks
  ILS: Human + Chimp UP                121,016 peaks
  ILS: Human + Bonobo UP               121,016 peaks
  Great Apes UP vs Macaque              81,809 peaks
  Old World UP vs Marmoset              64,058 peaks

Loading SNC BED file...
  SNC file loaded: 321,719 rows
  SNCs after fixed filter (>=0.9): 136,518
  136,518 fixed SNCs loaded


In [ ]:
# ── Main loop: heatmap + BED export for every cell type ───────────────────────

all_enrichment_rows = []

for cell_type in CELL_TYPES:
    print(f"\n{'='*65}")
    print(f"  {cell_type}")
    print(f"{'='*65}")

    # ── 1. Load accessibility matrices ────────────────────────────────────────
    print("Loading pseudobulk matrices...")
    acc_species_df, acc_donor_df, acc_donor_region_df = load_pseudobulk_matrices(
        cell_type, SPECIES, FRAG_DIR
    )

    if acc_species_df.empty:
        print(f"  [SKIP] {cell_type}: no pseudobulk data found")
        continue

    # Species-level matrix is always used for filtering (waterfall / posthoc)
    filter_acc_df = acc_species_df

    # Select plotting matrix
    if ACCESSIBILITY_VIEW == "species":
        acc_df = acc_species_df
        title_mode = "species-mean"
    elif ACCESSIBILITY_VIEW == "donor":
        acc_df = acc_donor_df
        title_mode = "donor"
    else:  # donor_region
        acc_df = acc_donor_region_df
        title_mode = "donor+region"

    # ── 2. Build evolutionary modules ─────────────────────────────────────────
    print("\nBuilding evolutionary modules...")
    regions, regions_all, total_candidates = build_regions(
        cell_type=cell_type,
        anno=anno,
        filter_acc_df=filter_acc_df,
        robust_dir=ROBUST_DIR,
        branch_dir=BRANCH_DIR,
        orth_cols=orth_cols,
        det_cols=det_cols,
        species=SPECIES,
        max_per_block=MAX_PER_BLOCK,
    )

    # ── 3. Build heatmap matrix ────────────────────────────────────────────────
    heatmap_plot_df, block_labels, block_sizes = build_heatmap_matrix(
        regions=regions,
        acc_df=acc_df,
        anno=anno,
        orth_cols=orth_cols,
        apply_row_zscore=ROW_ZSCORE,
    )
    print(f"\nHeatmap matrix: {heatmap_plot_df.shape[0]} peaks × "
          f"{heatmap_plot_df.shape[1]} columns")

    # ── 4. Plot & save heatmap ─────────────────────────────────────────────────
    ct_heatmap_dir = OUT_HEATMAPS / sanitize_name(cell_type)
    ct_heatmap_dir.mkdir(exist_ok=True)
    plot_evolutionary_heatmap(
        heatmap_plot_df=heatmap_plot_df,
        block_labels=block_labels,
        block_sizes=block_sizes,
        cell_type=cell_type,
        title_mode=title_mode,
        out_dir=ct_heatmap_dir,
        apply_row_zscore=ROW_ZSCORE,
    )

    # ── 5. Export BED files ────────────────────────────────────────────────────
    ct_bed_dir = OUT_BEDS / sanitize_name(cell_type)
    ct_bed_dir.mkdir(exist_ok=True)
    export_beds(
        regions_all=regions_all,
        anno=anno,
        out_dir=ct_bed_dir,
        cell_type=cell_type,
        coord_system="Human",
    )

    # ── 6. SNC enrichment ─────────────────────────────────────────────────────
    rows = compute_snc_enrichment(
        cell_type=cell_type,
        regions_all=regions_all,
        anno=anno,
        snc_bt=snc_bt,
        module_backgrounds=snc_backgrounds,  # contrast-specific backgrounds
        n_permutations=10_000,
    )
    all_enrichment_rows.extend(rows)
    for r in rows:
        fe_str = f"{r['fold_enrichment']:.2f}x" if not pd.isna(r['fold_enrichment']) else "n/a"
        fp_str = f"{r['fisher_pvalue']:.2e}"    if not pd.isna(r['fisher_pvalue'])    else "n/a"
        pp_str = f"{r['perm_pvalue']:.4f}"      if not pd.isna(r['perm_pvalue'])      else "n/a"
        d_str  = f"{r['density_fg_snc_per_kb']:.3f}" if not pd.isna(r['density_fg_snc_per_kb']) else "n/a"
        print(f"    SNC | {r['module']:<35} FE={fe_str:>7}  "
              f"fisher_p={fp_str}  perm_p={pp_str}  density={d_str}/kb")

print(f"\n{'='*65}")
print(f"Done. Processed {len(CELL_TYPES)} cell types.")


  Colonocytes
Loading pseudobulk matrices...
  Human: 6 Colonocytes pseudobulks (5494 cells)
  Bonobo: 1 Colonocytes pseudobulks (975 cells)
  Chimpanzee: 2 Colonocytes pseudobulks (463 cells)
  Gorilla: 1 Colonocytes pseudobulks (419 cells)
  Macaque: 2 Colonocytes pseudobulks (1620 cells)
  Marmoset: 2 Colonocytes pseudobulks (1407 cells)

Building evolutionary modules...
  Human-specific DNA:            1594 -> 50 shown
  Human UP vs All:               2869 -> 50 shown
  ILS: Human + Gorilla UP:     285 -> 50 shown
  ILS: Human + Chimp UP:      84 -> 50 shown
  ILS: Human + Bonobo UP:     236 -> 50 shown
  Great Apes UP vs Macaque:      1288 -> 50 shown
  Old World UP vs Marmoset:      2989 -> 50 shown

  Category                            Candidates   Shown
  ----------------------------------------------------------
  Human-specific DNA                        1594      50
  Human UP vs All                           2869      50
  ILS: Human + Gorilla UP                    285   

In [ ]:
# ── Compile enrichment table, apply FDR correction, save ─────────────────────

enrich_df = pd.DataFrame(all_enrichment_rows)

# BH FDR correction applied separately to Fisher and permutation p-values
def _apply_fdr(df, pval_col, padj_col):
    padj = np.full(len(df), np.nan)
    valid = df[pval_col].notna()
    if valid.sum() > 0:
        _, padj_vals, _, _ = multipletests(df.loc[valid, pval_col], method="fdr_bh")
        padj[valid.values] = padj_vals
    df[padj_col] = padj

_apply_fdr(enrich_df, "fisher_pvalue", "fisher_padj")
_apply_fdr(enrich_df, "perm_pvalue",   "perm_padj")

# Save full table
tsv_path = OUT_ENRICH / "SNC_enrichment_all_celltypes.tsv"
enrich_df.to_csv(tsv_path, sep="\t", index=False)
print(f"Enrichment table ({len(enrich_df)} rows) saved → {tsv_path}")

# Preview significant hits (by Fisher padj)
sig = enrich_df[enrich_df["fisher_padj"] < 0.05].sort_values("fold_enrichment", ascending=False)
print(f"\nSignificant enrichments (fisher_padj < 0.05): {len(sig)}")
display(sig[["cell_type", "module", "n_fg", "n_fg_snc", "pct_fg_snc",
             "pct_bg_snc", "fold_enrichment",
             "density_fg_snc_per_kb", "density_bg_snc_per_kb",
             "fisher_pvalue", "fisher_padj",
             "perm_pvalue", "perm_padj"]].head(20))

In [ ]:
# ── Plot enrichment dot plot ───────────────────────────────────────────────────

dot_path = OUT_ENRICH / "SNC_enrichment_dotplot.pdf"
plot_snc_enrichment_dotplot(
    enrich_df=enrich_df,
    out_path=dot_path,
    padj_col="fisher_padj",
    fe_col="fold_enrichment",
    pval_threshold=0.05,
)
print(f"\nDotplot saved → {dot_path}")